In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("ShopSphere Data Cleaning Started!")

ShopSphere Data Cleaning Started!


In [3]:
from pathlib import Path

DATA_PATH = Path("../data/raw")

customers = pd.read_csv(DATA_PATH / "customers.csv")
products = pd.read_csv(DATA_PATH / "products.csv")
orders = pd.read_csv(DATA_PATH / "orders.csv")
order_items = pd.read_csv(DATA_PATH / "order_items.csv")
payments = pd.read_csv(DATA_PATH / "payments.csv")

print("Datasets loaded successfully!")

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)

Datasets loaded successfully!
Customers: (20000, 10)
Products: (1000, 10)
Orders: (100000, 5)
Order Items: (300323, 5)
Payments: (100000, 6)


In [4]:
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)

Customers: (20000, 10)
Products: (1000, 10)
Orders: (100000, 5)
Order Items: (300323, 5)
Payments: (100000, 6)


In [5]:
# ==============================
# 1. MISSING VALUE CHECK
# ==============================

datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print(df.isnull().sum())


Customers
----------------------------------------
customer_id      0
first_name       0
last_name        0
gender           0
date_of_birth    0
city             0
state            0
pincode          0
signup_date      0
customer_type    0
dtype: int64

Products
----------------------------------------
product_id        0
product_name      0
category          0
subcategory       0
brand             0
selling_price     0
cost_price        0
stock_quantity    0
rating            0
launch_date       0
dtype: int64

Orders
----------------------------------------
order_id         0
customer_id      0
order_date       0
order_status     0
sales_channel    0
dtype: int64

Order Items
----------------------------------------
order_id        0
product_id      0
quantity        0
unit_price      0
discount_pct    0
dtype: int64

Payments
----------------------------------------
payment_id        0
order_id          0
payment_date      0
payment_method    0
payment_status    0
payment_amount  

In [6]:
# ==============================
# 2. DUPLICATE CHECK
# ==============================

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Customers: 0 duplicate rows
Products: 0 duplicate rows
Orders: 0 duplicate rows
Order Items: 0 duplicate rows
Payments: 0 duplicate rows


In [7]:
# ==============================
# 3. DATA TYPE CHECK
# ==============================

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.dtypes)


Customers
customer_id        str
first_name         str
last_name          str
gender             str
date_of_birth      str
city               str
state              str
pincode          int64
signup_date        str
customer_type      str
dtype: object

Products
product_id            str
product_name          str
category              str
subcategory           str
brand                 str
selling_price       int64
cost_price          int64
stock_quantity      int64
rating            float64
launch_date           str
dtype: object

Orders
order_id         str
customer_id      str
order_date       str
order_status     str
sales_channel    str
dtype: object

Order Items
order_id            str
product_id          str
quantity          int64
unit_price      float64
discount_pct      int64
dtype: object

Payments
payment_id            str
order_id              str
payment_date          str
payment_method        str
payment_status        str
payment_amount    float64
dtype: object


In [8]:
# ==============================
# 4. DATE TYPE CONVERSION
# ==============================

orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

payments["payment_date"] = pd.to_datetime(
    payments["payment_date"],
    errors="coerce"
)

print("Date conversion completed!")

print("\nOrders:")
print(orders["order_date"].dtype)

print("\nPayments:")
print(payments["payment_date"].dtype)

Date conversion completed!

Orders:
datetime64[us]

Payments:
datetime64[us]


In [9]:
# ==============================
# 5. ID VALIDATION
# ==============================

print("Customer IDs in Orders missing from Customers:")

missing_customers = ~orders["customer_id"].isin(customers["customer_id"])
print(missing_customers.sum())

print("\nProduct IDs in Order Items missing from Products:")

missing_products = ~order_items["product_id"].isin(products["product_id"])
print(missing_products.sum())

print("\nOrder IDs in Order Items missing from Orders:")

missing_orders = ~order_items["order_id"].isin(orders["order_id"])
print(missing_orders.sum())

print("\nOrder IDs in Payments missing from Orders:")

payment_order_check = ~payments["order_id"].isin(orders["order_id"])
print(payment_order_check.sum())

Customer IDs in Orders missing from Customers:
0

Product IDs in Order Items missing from Products:
0

Order IDs in Order Items missing from Orders:
0

Order IDs in Payments missing from Orders:
0


In [11]:
# ==============================
# 6. NUMERICAL VALUE VALIDATION
# ==============================

print("Order Items - Quantity:")
print(order_items["quantity"].describe())

print("\nOrder Items - Unit Price:")
print(order_items["unit_price"].describe())

print("\nOrder Items - Discount:")
print(order_items["discount_pct"].describe())

print("\nPayments - Payment Amount:")
print(payments["payment_amount"].describe())


Order Items - Quantity:
count    300323.000000
mean          1.511013
std           0.793775
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           4.000000
Name: quantity, dtype: float64

Order Items - Unit Price:
count    300323.000000
mean      12108.488727
std       17016.244378
min         110.000000
25%        1950.000000
50%        5190.000000
75%       15170.000000
max       79900.000000
Name: unit_price, dtype: float64

Order Items - Discount:
count    300323.000000
mean          8.141568
std           7.337454
min           0.000000
25%           0.000000
50%           5.000000
75%          10.000000
max          25.000000
Name: discount_pct, dtype: float64

Payments - Payment Amount:


count    100000.000000
mean      48421.435985
std       54352.929643
min           0.000000
25%       10727.000000
50%       30256.000000
75%       67095.250000
max      670559.000000
Name: payment_amount, dtype: float64


In [12]:
print("Quantity <= 0:", (order_items["quantity"] <= 0).sum())
print("Unit Price < 0:", (order_items["unit_price"] < 0).sum())
print("Discount < 0:", (order_items["discount_pct"] < 0).sum())
print("Payment Amount < 0:", (payments["payment_amount"] < 0).sum())

Quantity <= 0: 0
Unit Price < 0: 0
Discount < 0: 0
Payment Amount < 0: 0


In [13]:
# ==========================================
# 7. ID UNIQUENESS VALIDATION
# ==========================================

print("CUSTOMER ID DUPLICATES:",
      customers["customer_id"].duplicated().sum())

print("PRODUCT ID DUPLICATES:",
      products["product_id"].duplicated().sum())

print("ORDER ID DUPLICATES:",
      orders["order_id"].duplicated().sum())

print("PAYMENT ID DUPLICATES:",
      payments["payment_id"].duplicated().sum())

CUSTOMER ID DUPLICATES: 0
PRODUCT ID DUPLICATES: 0
ORDER ID DUPLICATES: 0
PAYMENT ID DUPLICATES: 0


In [14]:
# ==========================================
# 8. ORDER STATUS VALIDATION
# ==========================================

print("Order Status Values:")
print(orders["order_status"].value_counts())

print("\nUnique Statuses:")
print(orders["order_status"].unique())

Order Status Values:
order_status
Delivered     71869
Shipped       10056
Processing     8104
Cancelled      5019
Returned       4952
Name: count, dtype: int64

Unique Statuses:
<StringArray>
['Shipped', 'Delivered', 'Processing', 'Cancelled', 'Returned']
Length: 5, dtype: str


In [15]:
# ==========================================
# 9. SALES CHANNEL VALIDATION
# ==========================================

print("Sales Channel Values:")
print(orders["sales_channel"].value_counts())

print("\nUnique Channels:")
print(orders["sales_channel"].unique())

Sales Channel Values:
sales_channel
Website        44992
Mobile App     39918
Marketplace    15090
Name: count, dtype: int64

Unique Channels:
<StringArray>
['Website', 'Marketplace', 'Mobile App']
Length: 3, dtype: str


In [16]:
# ==========================================
# 10. PAYMENT VALIDATION
# ==========================================

print("Payment Methods:")
print(payments["payment_method"].value_counts())

print("\nPayment Status:")
print(payments["payment_status"].value_counts())

Payment Methods:
payment_method
UPI                 37858
Credit Card         21990
Debit Card          16005
Net Banking         10141
Cash on Delivery     7905
Wallet               6101
Name: count, dtype: int64

Payment Status:
payment_status
Successful    87329
Refunded       8458
Failed         4213
Name: count, dtype: int64


In [17]:
# ==========================================
# 11. DATE VALIDATION
# ==========================================

print("Order date range:")
print(orders["order_date"].min(), "to", orders["order_date"].max())

print("\nPayment date range:")
print(payments["payment_date"].min(), "to", payments["payment_date"].max())

Order date range:
2024-01-01 00:00:00 to 2025-12-31 00:00:00

Payment date range:
2024-01-02 00:00:00 to 2026-01-03 00:00:00


In [18]:
# ==========================================
# 11. DATE VALIDATION
# ==========================================

print("Order date range:")
print(orders["order_date"].min(), "to", orders["order_date"].max())

print("\nPayment date range:")
print(payments["payment_date"].min(), "to", payments["payment_date"].max())

Order date range:
2024-01-01 00:00:00 to 2025-12-31 00:00:00

Payment date range:
2024-01-02 00:00:00 to 2026-01-03 00:00:00


In [19]:
from pathlib import Path

PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Processed data folder ready!")

Processed data folder ready!


In [20]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()

print("Clean copies created successfully!")

Clean copies created successfully!


In [21]:
for name, df in {
    "Customers": customers_clean,
    "Products": products_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Payments": payments_clean
}.items():
    before = len(df)
    df.drop_duplicates(inplace=True)
    after = len(df)

    print(f"{name}: {before - after} duplicates removed")

Customers: 0 duplicates removed
Products: 0 duplicates removed
Orders: 0 duplicates removed
Order Items: 0 duplicates removed
Payments: 0 duplicates removed


In [22]:
print("Missing values before cleaning:")

for name, df in {
    "Customers": customers_clean,
    "Products": products_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Payments": payments_clean
}.items():
    
    missing = df.isnull().sum().sum()
    print(f"{name}: {missing}")

Missing values before cleaning:
Customers: 0
Products: 0
Orders: 0
Order Items: 0
Payments: 0


In [23]:
# Customers
for col in customers_clean.select_dtypes(include="object").columns:
    customers_clean[col] = customers_clean[col].str.strip()

# Products
for col in products_clean.select_dtypes(include="object").columns:
    products_clean[col] = products_clean[col].str.strip()

# Orders
for col in orders_clean.select_dtypes(include="object").columns:
    orders_clean[col] = orders_clean[col].str.strip()

# Order Items
for col in order_items_clean.select_dtypes(include="object").columns:
    order_items_clean[col] = order_items_clean[col].str.strip()

# Payments
for col in payments_clean.select_dtypes(include="object").columns:
    payments_clean[col] = payments_clean[col].str.strip()


print("Text columns standardized successfully!")

C:\Users\hp\AppData\Local\Temp\ipykernel_11040\550456579.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in customers_clean.select_dtypes(include="object").columns:
C:\Users\hp\AppData\Local\Temp\ipykernel_11040\550456579.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_g

Text columns standardized successfully!


In [24]:
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

payments_clean["payment_date"] = pd.to_datetime(
    payments_clean["payment_date"],
    errors="coerce"
)

print("Date columns cleaned successfully!")
print("Orders:", orders_clean["order_date"].dtype)
print("Payments:", payments_clean["payment_date"].dtype)

Date columns cleaned successfully!
Orders: datetime64[us]
Payments: datetime64[us]


In [25]:
print("FINAL CLEANING VALIDATION")
print("=" * 50)

clean_datasets = {
    "Customers": customers_clean,
    "Products": products_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Payments": payments_clean
}

for name, df in clean_datasets.items():
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing values:", df.isnull().sum().sum())
    
    print("Duplicate rows:", df.duplicated().sum())

FINAL CLEANING VALIDATION

Customers
Rows: 20000
Columns: 10
Missing values: 0
Duplicate rows: 0

Products
Rows: 1000
Columns: 10
Missing values: 0
Duplicate rows: 0

Orders
Rows: 100000
Columns: 5
Missing values: 0
Duplicate rows: 0

Order Items
Rows: 300323
Columns: 5
Missing values: 0
Duplicate rows: 0

Payments
Rows: 100000
Columns: 6
Missing values: 0
Duplicate rows: 0


In [26]:
customers_clean.to_csv(
    PROCESSED_PATH / "customers_clean.csv",
    index=False
)

products_clean.to_csv(
    PROCESSED_PATH / "products_clean.csv",
    index=False
)

orders_clean.to_csv(
    PROCESSED_PATH / "orders_clean.csv",
    index=False
)

order_items_clean.to_csv(
    PROCESSED_PATH / "order_items_clean.csv",
    index=False
)

payments_clean.to_csv(
    PROCESSED_PATH / "payments_clean.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!
